In [ ]:
import os, gc, json
import librosa
import numpy as np
import pandas as pd
import torch
import random
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from tqdm import tqdm
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer
)

sns.set(style="whitegrid")

In [ ]:
df = pd.read_csv("DataFrame/processed_df.csv")
df["path"] = df["path"].apply(lambda p: p.replace("\\", "/"))

datasets_all = df["dataset"].unique()
print("Datasets:", datasets_all)

#### Apply augmentation to training dataframe (2 datasets train, 2 datasets test)

In [ ]:
# ============================================================
# 2. VISUALIZATION HELPERS (emotion-only, cleaner)
# ============================================================
missing = [p for p in df["path"] if not os.path.exists(p)]
print("Missing files:", len(missing))
missing[:10]

def plot_distribution(df, title):
    plt.figure(figsize=(8,4))
    sns.countplot(data=df, x="emotion", palette="tab10")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 3. EMOTION-ONLY AUGMENTATION (separated per split)
# ============================================================
def aug_noise(audio):
    return audio + np.random.normal(0, 0.005, len(audio))

def aug_gain(audio):
    return audio * np.random.uniform(0.8, 1.2)

def aug_time_stretch(audio):
    if len(audio) < 2: return audio
    return librosa.effects.time_stretch(audio, rate=np.random.uniform(0.9, 1.1))

AUGS = [aug_noise, aug_gain, aug_time_stretch]

# ------------------------------------------------------------
# Augment and balance train_df
# ------------------------------------------------------------
def augment_and_balance(train_df):
    emotion_counts = train_df["emotion"].value_counts()
    max_count = emotion_counts.max()

    print("Before balancing:\n", emotion_counts)

    new_rows = []
    audio_cache = {}

    for emotion, count in emotion_counts.items():
        needed = max_count - count
        if needed == 0: continue

        src_rows = train_df[train_df["emotion"] == emotion]

        for i in tqdm(range(needed), desc=f"Aug {emotion}"):
            row = src_rows.sample(1).iloc[0]
            audio, sr = librosa.load(row["path"], sr=16000)
            aug_fn = random.choice(AUGS)
            aug_audio = aug_fn(audio)

            vpath = f"memory://{emotion}_{i}"
            audio_cache[vpath] = aug_audio

            new_rows.append({
                "dataset": row["dataset"],
                "path": vpath,
                "emotion": emotion
            })

    df_new = pd.concat([train_df, pd.DataFrame(new_rows)], ignore_index=True)
    df_new = df_new.sample(frac=1, random_state=42).reset_index(drop=True)

    print("After balancing:\n", df_new["emotion"].value_counts())
    return df_new, audio_cache

In [ ]:
# ============================================================
# 4. HF Dataset preprocessing
# ============================================================

def preprocess_fn(batch, extractor, audio_cache):
    audios = []
    for p in batch["path"]:
        if str(p).startswith("memory://"):
            audios.append(audio_cache[p])
        else:
            audios.append(librosa.load(p, sr=16000)[0])

    inputs = extractor(audios, sampling_rate=16000,
                       return_tensors="pt", padding=True)
    inputs["labels"] = batch["label"]
    return inputs

In [ ]:
# ============================================================
# 5. TRAINING FUNCTION (Wav2Vec2 for 2v2 split)
# ============================================================
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

def run_training(train_df, test_df, name, audio_cache):
    print(f"\nTraining 2v2 on {name}")

    # Encode labels (plain Python types)
    class_labels = [str(l) for l in sorted(df["emotion"].unique())]
    label2id = {str(l): int(i) for i, l in enumerate(class_labels)}
    id2label = {int(i): str(l) for l, i in label2id.items()}

    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df["emotion"] = train_df["emotion"].astype(str)
    test_df["emotion"] = test_df["emotion"].astype(str)
    train_df["label"] = train_df["emotion"].map(label2id)
    test_df["label"] = test_df["emotion"].map(label2id)

    train_df, val_df = train_test_split(
        train_df, test_size=0.1, random_state=42, stratify=train_df["emotion"]
    )

    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df),
        "validation": Dataset.from_pandas(val_df),
        "test": Dataset.from_pandas(test_df)
    })

    extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

    ds = ds.map(
        lambda batch: preprocess_fn(batch, extractor, audio_cache),
        batched=True,
        remove_columns=ds["train"].column_names
    )

    for split in ds:
        ds[split].set_format(type="torch", columns=["input_values", "labels"])

    model = AutoModelForAudioClassification.from_pretrained(
        "facebook/wav2vec2-base",
        num_labels=len(class_labels),
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True
    )

    model.gradient_checkpointing_enable()

    args = TrainingArguments(
        output_dir=f"Models/2v2_{name}",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        metric_for_best_model="accuracy",
        fp16=torch.cuda.is_available(),
        logging_strategy="epoch",
        report_to="none",
        save_only_model=True
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],
        compute_metrics=compute_metrics,
        processing_class=extractor
    )

    trainer.train()
    trainer.save_model(f"Models/2v2_{name}/best_model")

    return f"Models/2v2_{name}"

In [ ]:
# ============================================================
# 6. EVALUATION (Confusion Matrix, Metrics, Report)
# ============================================================
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

def evaluate_model(model_dir, test_df):

    model = AutoModelForAudioClassification.from_pretrained(model_dir + "/best_model")
    extractor = AutoFeatureExtractor.from_pretrained(model_dir + "/best_model")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()

    # Use model config for label decoding and ensure test labels are strings
    class_labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
    test_df = test_df.copy()
    test_df["emotion"] = test_df["emotion"].astype(str)

    y_true, y_pred = [], []

    for _, row in test_df.iterrows():
        audio = librosa.load(row["path"], sr=16000)[0]

        inputs = extractor([audio], sampling_rate=16000, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k,v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits
            pred_idx = torch.argmax(logits, dim=-1).cpu().item()
            pred_label = class_labels[pred_idx]

        y_true.append(row["emotion"])
        y_pred.append(pred_label)

    # ---------------- METRICS ----------------
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    metrics = {
        "accuracy": round(acc, 3),
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
    }

    # ---------------- SAVE METRICS ----------------
    with open(model_dir + "/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    pd.DataFrame([metrics]).to_csv(model_dir + "/metrics.csv", index=False)

    # ---------------- SAVE REPORT ----------------
    report = classification_report(y_true, y_pred, output_dict=True)
    report_df = pd.DataFrame(report).round(3)
    report_df.to_csv(model_dir + "/classification_report.csv")

    # ---------------- SAVE CONFUSION MATRIX ----------------
    cm = confusion_matrix(y_true, y_pred, labels=class_labels)
    fig, ax = plt.subplots(figsize=(8,6))
    ConfusionMatrixDisplay(cm, display_labels=class_labels).plot(
        ax=ax, cmap="Blues", xticks_rotation=45, values_format="d")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(model_dir + "/confusion_matrix.png", dpi=200)
    plt.close()

    # ---------------- SAVE PREDICTIONS ----------------
    pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred
    }).to_csv(model_dir + "/predictions.csv", index=False)

    return metrics

In [ ]:
# ========================
# 7. RUN 2v2 EXPERIMENT (2 datasets train, 2 datasets test)
# ========================
summary_results = []

# Generate all combinations of 2 datasets for testing
test_combinations = list(combinations(datasets_all, 2))

for test_pair in test_combinations:
    # Get the other 2 datasets for training
    train_pair = [ds for ds in datasets_all if ds not in test_pair]
    
    print("\n=========================================")
    print(f"Train: {train_pair}")
    print(f"Test: {list(test_pair)}")
    print("=========================================")

    # Clear caches before each split
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 1. Create 2v2 split
    train_df = df[df["dataset"].isin(train_pair)].copy()
    test_df = df[df["dataset"].isin(test_pair)].copy()

    # Create name for this split
    train_name = "+".join(sorted(train_pair))
    test_name = "+".join(sorted(test_pair))
    split_name = f"train_{train_name}_test_{test_name}"

    plot_distribution(train_df, f"Before Aug — Train: {train_name}")

    # 2. Augmentation + balancing
    train_bal, audio_cache = augment_and_balance(train_df)

    plot_distribution(train_bal, f"After Aug — Train: {train_name}")

    # 3. Train model
    model_dir = run_training(train_bal, test_df, split_name, audio_cache)

    # 4. Evaluate and save metrics, cm, reports
    metrics = evaluate_model(model_dir, test_df)
    metrics["train_datasets"] = train_name
    metrics["test_datasets"] = test_name
    summary_results.append(metrics)

    # clear memory
    audio_cache.clear()
    del audio_cache
    gc.collect()

# ========================
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv("Models/2v2_summary.csv", index=False)

print("\n============= 2v2 SUMMARY =============")
print(summary_df)